In [1]:
#Amo demo
#GV 19.9.2025 + FM 22/10/2025 + FM 05.11.2025
import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo_with_doublings_multihot

import pandas as pd
import numpy as np
from collections import defaultdict

/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-11-14 18:00:10.813432: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# amo_with_doublings
# Args:
  #      filein (str): Path to the source MIDI file to learn orchestration style from.
  #      fileout (str): Path to the target MIDI file to be orchestrated.
  #      ytarget (str, optional): The target variable for the model ('track-channel' or 'program').
  #                               Defaults to "track-channel".
  #      model (str, optional): The name of the machine learning model to use for orchestration.
  #     ("XGBoost","RandomForest", "DecisionTree", "NearestNeighbors", "MLP3", "NaiveBayes", "MLP1", "AdaBoost", "LSTMClassifier", "TransformerClassifier")
  #                             Defaults to "XGBoost".

In [3]:
def transpose(note, inverse=False, n_semitones=12):
    # Example: transpose pitch
    if inverse:
        n_semitones = - n_semitones
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [4]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]

In [5]:
filein='midis/sugar-plum-fairy_orch.mid'
fileout='midis/fur-elise.mid'
amo_with_doublings_multihot(filein,fileout,model="XGBoost", tol=0.2, transformations=transformations)

Learning orchestration style from: midis/sugar-plum-fairy_orch.mid
Mapping: [[1 '3 Flutes' 1 73]
 [1 '3 Flutes' 2 73]
 [1 '3 Flutes' 3 73]
 [2 '2 Oboes' 4 68]
 [3 'English Horn' 5 69]
 [4 '2 Clarinets in A' 6 71]
 [4 '2 Clarinets in A' 7 71]
 [5 'Bass Clarinet in Bb' 2 71]
 [6 '2 Bassoons' 8 70]
 [6 '2 Bassoons' 10 70]
 [7 '4 Horns in F' 11 60]
 [7 '4 Horns in F' 12 60]
 [8 'Celesta' 0 8]
 [9 'Violin I' 13 45]
 [9 'Violin I' 13 48]
 [10 'Violin II' 14 45]
 [10 'Violin II' 14 48]
 [11 'Viola' 15 48]
 [11 'Viola' 15 45]
 [12 'Violoncello' 0 48]
 [12 'Violoncello' 0 45]
 [13 'Contrabass' 1 45]
 [13 'Contrabass' 1 48]]
Building reduced dataset
{'transformed': 337, 'direct': 50, 'none': 1438}
Original size: (1825, 15)
Dropping [1766, 1767, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1552, 1780, 1781, 1554, 1782, 1783, 1784, 1785, 470, 1557, 1786, 486, 487, 1787, 492, 493, 498, 499, 1788, 1560, 515, 1561, 1699, 1789, 1562, 1790, 523, 526, 1563, 1701, 1791, 1792, 1